In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [2]:
def fetch_creditcard(X_y_split: bool = False
                     ) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    response = requests.get("http://pipeline:8000/dataset/creditcard-churn", params={"X_y_split": X_y_split})
    payload = response.json()
    
    if X_y_split:
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"])
        return X, y
    
    df = pd.DataFrame(payload["data"], index=payload["index"])
    
    return df

In [3]:
# 데이터 로드
bank_df = fetch_creditcard()
bank_df = bank_df.drop('creditcard_churn_id', axis=1)
print(bank_df.columns)

Index(['churn', 'age', 'gender', 'dependents', 'education_id', 'marital_id',
       'income_id', 'card_type_id', 'relationship_months', 'product_count',
       'inactive_months', 'contact_count', 'credit_limit', 'revolving_balance',
       'available_credit', 'amount_change', 'count_change',
       'transaction_amount', 'transaction_count', 'utilization_ratio'],
      dtype='object')


In [4]:
# 목표 데이터 분리
churn_df = bank_df["churn"]
print(churn_df.value_counts())

churn
0    8561
1    1666
Name: count, dtype: int64


In [5]:
# feature 데이터 분리
feature_df = bank_df[[k for k in bank_df.columns if k != "churn"]]
print(feature_df.columns)

Index(['age', 'gender', 'dependents', 'education_id', 'marital_id',
       'income_id', 'card_type_id', 'relationship_months', 'product_count',
       'inactive_months', 'contact_count', 'credit_limit', 'revolving_balance',
       'available_credit', 'amount_change', 'count_change',
       'transaction_amount', 'transaction_count', 'utilization_ratio'],
      dtype='object')


In [6]:
from sklearn.model_selection import train_test_split

# train - test split을 활용해서 학습/테스트 데이터 분리
train_X, test_X, train_y, test_y = train_test_split(
    feature_df,
    churn_df,
    test_size=0.2,
    stratify=churn_df,
    random_state=42
)

print(train_X.describe())

               age       gender   dependents  education_id   marital_id  \
count  8181.000000  8181.000000  8181.000000   8181.000000  8181.000000   
mean     46.315365     0.468280     2.349102      3.654810     1.842073   
std       8.064247     0.499023     1.299942      1.910247     0.863124   
min      25.000000     0.000000     0.000000      1.000000     1.000000   
25%      41.000000     0.000000     1.000000      2.000000     1.000000   
50%      46.000000     0.000000     2.000000      4.000000     2.000000   
75%      52.000000     1.000000     3.000000      4.000000     2.000000   
max      73.000000     1.000000     5.000000      7.000000     4.000000   

         income_id  card_type_id  relationship_months  product_count  \
count  8181.000000   8181.000000          8181.000000    8181.000000   
mean      2.745508      1.104633            35.922626       3.806258   
std       1.712108      0.410503             8.138261       1.551792   
min       1.000000      1.000000    

In [7]:
def print_split_report(X_train, X_test, y_train, y_test):
    # [📊 요약 출력 - 팀 공통 규격]
    print(f"\n{'Train / Test Split Summary':^50}")
    print("="*50)
    print(f"X_train shape : {X_train.shape}")
    print(f"X_test  shape : {X_test.shape}")
    print("-"*50)
    print(f"y_train shape : {y_train.shape}")
    print(f"y_test  shape : {y_test.shape}")
    print("-"*50)
    print("Target Distribution (Train)")
    print(y_train.value_counts().to_string())
    print("-"*50)
    print("Target Distribution (Test)")
    print(y_test.value_counts().to_string())
    print("-"*50)
    print("Train Ratio")
    print(y_train.value_counts(normalize=True).to_string())
    print("-"*50)
    print("Test Ratio")
    print(y_test.value_counts(normalize=True).to_string())
    print("="*50)

print_split_report(train_X, test_X, train_y, test_y)


            Train / Test Split Summary            
X_train shape : (8181, 19)
X_test  shape : (2046, 19)
--------------------------------------------------
y_train shape : (8181,)
y_test  shape : (2046,)
--------------------------------------------------
Target Distribution (Train)
churn
0    6848
1    1333
--------------------------------------------------
Target Distribution (Test)
churn
0    1713
1     333
--------------------------------------------------
Train Ratio
churn
0    0.837061
1    0.162939
--------------------------------------------------
Test Ratio
churn
0    0.837243
1    0.162757


In [8]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.multiclass import type_of_target


def print_report(y_true, y_pred, y_proba, classes=None, average="macro", dataset_name="Dataset"):
    target_type = type_of_target(y_true)

    result = {
        "dataset": dataset_name,
        "target_type": target_type,
        "accuracy": accuracy_score(y_true, y_pred),
    }

    # y_proba shape 보정
    y_proba = np.asarray(y_proba)

    if target_type == "binary":
        result["recall"] = recall_score(y_true, y_pred)
        result["f1"] = f1_score(y_true, y_pred)

        # (n_samples, 2) 이면 positive class 확률만 사용
        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            y_score = y_proba[:, 1]
        # 이미 (n_samples,) 형태면 그대로 사용
        elif y_proba.ndim == 1:
            y_score = y_proba
        else:
            print(f"Binary Classification Shape Error: y_proba.shape = {y_proba.shape}")
            return

        result["roc_auc"] = roc_auc_score(y_true, y_score)
        result["pr_auc"] = average_precision_score(y_true, y_score)
        result["precision_score"] = precision_score(y_true, y_pred)

    elif target_type == "multiclass":
        result["recall"] = recall_score(y_true, y_pred, average=average)
        result["f1"] = f1_score(y_true, y_pred, average=average)

        if y_proba.ndim != 2:
            raise ValueError(f"Multiclass classification에서는 y_proba가 2차원이어야 합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average=average
        )

        if classes is None:
            classes = np.unique(y_true)

        y_true_bin = label_binarize(y_true, classes=classes)
        result["pr_auc"] = average_precision_score(
            y_true_bin,
            y_proba,
            average=average
        )
        
        result["precision_score"] = precision_score(y_true, y_pred, average=average)

    else:
        raise ValueError(f"지원하지 않는 target type입니다: {target_type}")

    print("=" * 20, f"{result["dataset"]}", "="*20)
    print(f"{"target type":>20}: {result["target_type"]}")
    for k in [key for key in result if key not in ["dataset", "target_type"]]:
        print(f"{k:>20}: {result[k]:.6f}")


In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=5000,
    validation_fraction=0.2,
    early_stopping=True,
    n_iter_no_change=10
)

hgb_clf.fit(train_X, train_y)

train_pred = hgb_clf.predict(train_X)
test_pred = hgb_clf.predict(test_X)

train_proba = hgb_clf.predict_proba(train_X)
test_proba = hgb_clf.predict_proba(test_X)

print_report(
    y_true=train_y,
    y_pred=train_pred,
    y_proba=train_proba,
    classes=hgb_clf.classes_,
    dataset_name="Train"
)

print_report(
    y_true=test_y,
    y_pred=test_pred,
    y_proba=test_proba,
    classes=hgb_clf.classes_,
    dataset_name="Test"
)

==================== Train ====================
         target type: binary
            accuracy: 0.993522
              recall: 0.975994
                  f1: 0.980038
             roc_auc: 0.998311
              pr_auc: 0.994671
     precision_score: 0.984115
==================== Test ====================
         target type: binary
            accuracy: 0.972141
              recall: 0.879880
                  f1: 0.911353
             roc_auc: 0.994176
              pr_auc: 0.974911
     precision_score: 0.945161


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[800, 1000, 1200],    # 약학습기 개수
    'validation_fraction':[0.1, 0.2, 0.3], # 일정 비율을 검증용 데이터로 사용
    'n_iter_no_change':[15, 20, 25], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='average_precision')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 800, 'n_iter_no_change': 20, 'validation_fraction': 0.1}
0.9904391065656647
HistGradientBoostingClassifier(early_stopping=True, max_iter=800,
                               n_iter_no_change=20, random_state=42)


In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True, n_iter_no_change=20)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[200, 500, 800],    # 약학습기 개수
    'validation_fraction':[0.05, 0.1, 0.15], # 일정 비율을 검증용 데이터로 사용
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 200, 'validation_fraction': 0.05}
0.9906899170531336
HistGradientBoostingClassifier(early_stopping=True, max_iter=200,
                               n_iter_no_change=20, random_state=42,
                               validation_fraction=0.05)


In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True, n_iter_no_change=20)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[100, 200, 300],    # 약학습기 개수
    'validation_fraction':[0.03, 0.05, 0.08], # 일정 비율을 검증용 데이터로 사용
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 100, 'validation_fraction': 0.05}
0.9907018964809124
HistGradientBoostingClassifier(early_stopping=True, n_iter_no_change=20,
                               random_state=42, validation_fraction=0.05)


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.05
)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[50, 100, 150]    # 약학습기 개수
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 150}
0.9907216292377203
HistGradientBoostingClassifier(early_stopping=True, max_iter=150,
                               n_iter_no_change=20, random_state=42,
                               validation_fraction=0.05)


In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.05
)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[120, 150, 180]    # 약학습기 개수
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 150}
0.9907216292377203
HistGradientBoostingClassifier(early_stopping=True, max_iter=150,
                               n_iter_no_change=20, random_state=42,
                               validation_fraction=0.05)


In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    max_iter=150,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.05
)

hgb_clf.fit(train_X, train_y)

train_pred = hgb_clf.predict(train_X)
test_pred = hgb_clf.predict(test_X)

train_proba = hgb_clf.predict_proba(train_X)
test_proba = hgb_clf.predict_proba(test_X)

print_report(
    y_true=train_y,
    y_pred=train_pred,
    y_proba=train_proba,
    classes=hgb_clf.classes_,
    dataset_name="Train"
)

print_report(
    y_true=test_y,
    y_pred=test_pred,
    y_proba=test_proba,
    classes=hgb_clf.classes_,
    dataset_name="Test"
)

==================== Train ====================
         target type: binary
            accuracy: 0.993522
              recall: 0.975994
                  f1: 0.980038
             roc_auc: 0.998311
              pr_auc: 0.994671
     precision_score: 0.984115
==================== Test ====================
         target type: binary
            accuracy: 0.972141
              recall: 0.879880
                  f1: 0.911353
             roc_auc: 0.994176
              pr_auc: 0.974911
     precision_score: 0.945161
